### Задача 1

Асинхронный чат

Реализовать чат без графического интерфейса, который позволит обмениваться сообщениями только между клиентом и сервером. Чат должен быть асинхронным

Клиент и сервер можно запустить одновременно в различных окнах терминала

### Задача 2

Напишите фцнкцию, которая сможет за минимальное время собирать информацию с 1-го сервиса по 100_000 запросам

В примере представлен синхронный вариант

Сервис 'http://localhost:8003' вы можете запустить, он в файле `service.py`

In [ ]:
import requests
import json

params = range(100_000)

url = 'http://localhost:8003'

class Composer:
    def __init__(self):
        self.result = 0
    
    def add(self, data):
        self.result += data['data']


def collector(params: list[str]) -> int:
    composer = Composer()
    for i in params:
        resp = requests.get(f'{url}/{i}')
        composer.add(resp.json())
    
    return composer.result

In [9]:
import asyncio
import httpx

params = range(100_000)

url = 'http://localhost:8003'


class Composer:
    def __init__(self):
        self.result = 0

    def add(self, data):
        self.result += data['data']


async def async_function(client, i, sem):
    async with sem:
        resp = await client.get(f"{url}/{i}")
        return resp.json()


async def collector(params: list[str]) -> int:
    composer = Composer()
    results = []  
    sem = asyncio.Semaphore(1000)
    async with httpx.AsyncClient() as client:
        coros = []
        for i in params:
            coros.append(async_function(client, i, sem))      
        results = await asyncio.gather(*coros)
        
        for data in results:
            composer.add(data)
    
    return composer.result